# Database and Data-Warehouse Design

Create an operational SQLite database and transform it into a star-schema analytics warehouse.


In [ ]:
import sqlite3
from pathlib import Path

OP_DB = Path("operational_flights.db")
DW_DB = Path("flight_warehouse.db")


In [ ]:
with sqlite3.connect(OP_DB) as con:
    con.executescript('''
    DROP TABLE IF EXISTS flights;
    DROP TABLE IF EXISTS airlines;
    DROP TABLE IF EXISTS airports;

    CREATE TABLE airlines (
        airline_id INTEGER PRIMARY KEY,
        airline_name TEXT NOT NULL UNIQUE
    );

    CREATE TABLE airports (
        airport_code TEXT PRIMARY KEY,
        city TEXT NOT NULL
    );

    CREATE TABLE flights (
        flight_id INTEGER PRIMARY KEY,
        airline_id INTEGER NOT NULL,
        origin_code TEXT NOT NULL,
        destination_code TEXT NOT NULL,
        flight_date TEXT NOT NULL,
        delay_minutes INTEGER NOT NULL CHECK(delay_minutes >= 0),
        FOREIGN KEY (airline_id) REFERENCES airlines(airline_id),
        FOREIGN KEY (origin_code) REFERENCES airports(airport_code),
        FOREIGN KEY (destination_code) REFERENCES airports(airport_code)
    );
    ''')

    con.executemany("INSERT INTO airlines VALUES (?, ?)", [
        (1, "Air Canada"), (2, "WestJet"), (3, "Porter")
    ])
    con.executemany("INSERT INTO airports VALUES (?, ?)", [
        ("YYZ", "Toronto"), ("YVR", "Vancouver"), ("YUL", "Montreal"), ("YYC", "Calgary")
    ])
    con.executemany("INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?)", [
        (1,1,"YYZ","YVR","2026-01-04",20),
        (2,2,"YYC","YYZ","2026-01-04",0),
        (3,3,"YUL","YYZ","2026-01-05",35),
        (4,1,"YVR","YYZ","2026-01-05",10),
        (5,2,"YYZ","YYC","2026-01-06",55),
    ])


In [ ]:
with sqlite3.connect(OP_DB) as src:
    rows = src.execute('''
        SELECT f.flight_id, a.airline_name, f.origin_code, f.destination_code,
               f.flight_date, f.delay_minutes
        FROM flights f
        JOIN airlines a ON f.airline_id = a.airline_id
    ''').fetchall()

rows


In [ ]:
with sqlite3.connect(DW_DB) as dw:
    dw.executescript('''
    DROP TABLE IF EXISTS fact_flight;
    DROP TABLE IF EXISTS dim_airline;
    DROP TABLE IF EXISTS dim_route;
    DROP TABLE IF EXISTS dim_date;

    CREATE TABLE dim_airline (
        airline_key INTEGER PRIMARY KEY,
        airline_name TEXT NOT NULL
    );
    CREATE TABLE dim_route (
        route_key INTEGER PRIMARY KEY,
        origin_code TEXT NOT NULL,
        destination_code TEXT NOT NULL
    );
    CREATE TABLE dim_date (
        date_key INTEGER PRIMARY KEY,
        full_date TEXT NOT NULL,
        year INTEGER NOT NULL,
        month INTEGER NOT NULL,
        day INTEGER NOT NULL
    );
    CREATE TABLE fact_flight (
        flight_key INTEGER PRIMARY KEY,
        airline_key INTEGER NOT NULL,
        route_key INTEGER NOT NULL,
        date_key INTEGER NOT NULL,
        delay_minutes INTEGER NOT NULL,
        delayed_15min INTEGER NOT NULL
    );
    ''')

    airlines = sorted({r[1] for r in rows})
    routes = sorted({(r[2], r[3]) for r in rows})
    dates = sorted({r[4] for r in rows})

    airline_map = {name:i+1 for i,name in enumerate(airlines)}
    route_map = {route:i+1 for i,route in enumerate(routes)}
    date_map = {date:i+1 for i,date in enumerate(dates)}

    dw.executemany("INSERT INTO dim_airline VALUES (?, ?)", [(k,v) for v,k in airline_map.items()])
    dw.executemany("INSERT INTO dim_route VALUES (?, ?, ?)", [(k,*v) for v,k in route_map.items()])
    dw.executemany("INSERT INTO dim_date VALUES (?, ?, ?, ?, ?)", [
        (k, d, int(d[:4]), int(d[5:7]), int(d[8:10])) for d,k in date_map.items()
    ])
    dw.executemany("INSERT INTO fact_flight VALUES (?, ?, ?, ?, ?, ?)", [
        (r[0], airline_map[r[1]], route_map[(r[2],r[3])], date_map[r[4]], r[5], int(r[5] >= 15))
        for r in rows
    ])


In [ ]:
with sqlite3.connect(DW_DB) as dw:
    report = dw.execute('''
        SELECT a.airline_name, COUNT(*) AS flights,
               ROUND(AVG(f.delay_minutes), 2) AS avg_delay,
               SUM(f.delayed_15min) AS delayed_flights
        FROM fact_flight f
        JOIN dim_airline a ON f.airline_key = a.airline_key
        GROUP BY a.airline_name
        ORDER BY avg_delay DESC
    ''').fetchall()

report
